Creating a database from a folder structure, as in your `insert_folder_structure()` function, offers a variety of **practical benefits**, especially for **exploration, automation, analytics**, and **system design**. Below is a structured explanation of its **purpose and advantages**.

---

## #data #engineering #data\_storage #analysis #software

## 📦 Purpose of Building a Folder-File SQLite Database

### ✅ 1. **Searchability & Querying**

Instead of traversing the file system repeatedly with `os.walk`, you can:

* Search by file name, type, size, folder path.
* Query recently modified files.
* Filter by file extension or directory depth.

**Example:**

```sql
SELECT path FROM files WHERE extension = '.csv' AND size > 100000;
```

---

### ✅ 2. **Metadata Analysis**

Storing file and folder metadata in a table allows:

* Size analysis per folder or extension.
* Change tracking (modified vs created time).
* Statistical profiling (e.g., most common file types).

---

### ✅ 3. **System Monitoring & Auditing**

This provides a snapshot of the filesystem at a point in time.

* Compare snapshots to detect changes.
* Use for compliance (e.g., are files where they should be?).
* Identify duplicates, stale data, or orphaned files.

---

### ✅ 4. **Visualization and Reporting**

With SQL + pandas, you can:

* Build dashboards (e.g., using Streamlit or Dash).
* Visualize directory sizes, file type distributions, growth over time.

---

### ✅ 5. **Performance**

Traversing millions of files with `os.walk()` is expensive and slow. A SQLite3 database:

* Enables faster lookups and filters.
* Avoids redundant scans.
* Can be indexed (e.g., on file path or extension) for efficiency.

---

### ✅ 6. **Cross-Platform Data Portability**

* SQLite databases are portable, self-contained, and easy to integrate with other tools (e.g., Python, R, Excel).
* Can be embedded into larger systems or exposed via APIs.

---

### ✅ 7. **Automation Tooling**

You can:

* Build tooling that watches for file changes (with snapshot diffs).
* Generate reports on folder hygiene (e.g., unused files).
* Trigger alerts if sensitive files appear or grow unexpectedly.

---

## ⚠️ Potential Limitations

| Limitation                            | Mitigation                                                    |
| ------------------------------------- | ------------------------------------------------------------- |
| Data can go stale                     | Re-scan periodically or on change detection.                  |
| Does not include file contents        | Add indexing with full-text search tools if needed.           |
| SQLite not good for concurrent writes | Fine for read-heavy use cases or migrate to PostgreSQL later. |

---

## 🧠 Exploratory Questions

* Do you want to analyze growth over time (e.g., snapshots per week)?
* Should files be linked to projects, owners, or tags?
* Do you want to enrich the data with content-based info (e.g., file hash, text previews)?

Let me know if you'd like:

* ERD diagrams for the schema.
* Example analytics notebooks (e.g., file heatmaps).
* Extension to include symlink tracking or versioning.


In [1]:
import os
import sqlite3
from pathlib import Path
from datetime import datetime


In [2]:
# Get the path of the notebook
current_notebook = Path(__file__).resolve() if '__file__' in globals() else Path().resolve()

# Traverse up to DE_Tools
ROOT_DIR = current_notebook.parents[3]  # Go up 4 levels to get to DE_Tools

# Define where to store the SQLite database
DB_PATH = current_notebook/ "folder_structure.db"

print(f"Scanning root: {ROOT_DIR}")
print(f"Saving DB to:  {DB_PATH}")


Scanning root: C:\Users\RhysL\Desktop\DE_Tools
Saving DB to:  C:\Users\RhysL\Desktop\DE_Tools\Explorations\Other\Folder-Tools\Folder-DB\folder_structure.db


In [3]:

# --- Connect to SQLite3 database ---
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

In [4]:

# --- Create schema ---
cursor.execute('''
CREATE TABLE IF NOT EXISTS folders (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    path TEXT NOT NULL,
    parent_id INTEGER,
    FOREIGN KEY(parent_id) REFERENCES folders(id)
)
''')

cursor.execute('''
CREATE TABLE IF NOT EXISTS files (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    path TEXT NOT NULL,
    folder_id INTEGER,
    extension TEXT,
    size INTEGER,
    modified_time TEXT,
    created_time TEXT,
    FOREIGN KEY(folder_id) REFERENCES folders(id)
)
''')

conn.commit()


In [5]:
def insert_folder_structure(base_path: Path):
    folder_id_map = {}

    def scan_directory(current_path: Path):
        if current_path.name.startswith("."):
            return  # Skip hidden directories

        parent_path = current_path.parent
        parent_id = folder_id_map.get(parent_path.as_posix())

        # Insert current folder
        cursor.execute(
            "INSERT INTO folders (name, path, parent_id) VALUES (?, ?, ?)",
            (current_path.name, current_path.as_posix(), parent_id)
        )
        folder_id = cursor.lastrowid
        folder_id_map[current_path.as_posix()] = folder_id

        try:
            entries = list(os.scandir(current_path))
        except PermissionError:
            return  # Skip folders you can't access

        for entry in entries:
            if entry.name.startswith("."):
                continue  # Skip hidden files and folders

            full_path = current_path / entry.name

            if entry.is_dir(follow_symlinks=False):
                scan_directory(full_path)
            elif entry.is_file(follow_symlinks=False):
                try:
                    stat = entry.stat()
                except FileNotFoundError:
                    continue  # Skip broken symlinks or transient files

                cursor.execute(
                    '''INSERT INTO files 
                       (name, path, folder_id, extension, size, modified_time, created_time) 
                       VALUES (?, ?, ?, ?, ?, ?, ?)''',
                    (
                        entry.name,
                        full_path.as_posix(),
                        folder_id,
                        full_path.suffix,
                        stat.st_size,
                        datetime.fromtimestamp(stat.st_mtime).isoformat(),
                        datetime.fromtimestamp(stat.st_ctime).isoformat()
                    )
                )

    # Begin scan
    scan_directory(base_path)
    conn.commit()


In [6]:
# --- Run the process ---
if __name__ == "__main__":
    if not ROOT_DIR.exists():
        print(f"Root folder does not exist: {ROOT_DIR}")
    else:
        print(f"Scanning: {ROOT_DIR}")
        insert_folder_structure(ROOT_DIR)
        print("Done.")
        print(f"Database saved to: {DB_PATH.resolve()}")


Scanning: C:\Users\RhysL\Desktop\DE_Tools
Done.
Database saved to: C:\Users\RhysL\Desktop\DE_Tools\Explorations\Other\Folder-Tools\Folder-DB\folder_structure.db


In [7]:
cursor.close()

## SQL Queries

In [8]:
import sqlite3
import pandas as pd
from pathlib import Path

In [15]:

# Path to your SQLite database
DB_PATH = Path("folder_structure.db")
conn = sqlite3.connect(DB_PATH)

def run_query(query, params=None, description=None):
    if description:
        print(f"--- {description} ---\n")
    df = pd.read_sql(query, conn, params=params)
    return df


In [16]:
# --- Basic queries ---

run_query("""
SELECT 
    (SELECT COUNT(*) FROM folders) AS total_folders,
    (SELECT COUNT(*) FROM files) AS total_files;
""", description="1. Count total folders and files")


--- 1. Count total folders and files ---



,total_folders,total_files
0,2577,22996


In [17]:
run_query("""
SELECT id, name, path 
FROM folders 
WHERE parent_id IS NULL;
""", description="2. List top-level folders")

--- 2. List top-level folders ---



,id,name,path
0,1,DE_Tools,C:/Users/RhysL/Desktop/DE_Tools


In [18]:
run_query("""
SELECT f.name, f.path, COUNT(*) AS file_count
FROM folders f
JOIN files fi ON f.id = fi.folder_id
GROUP BY f.id
ORDER BY file_count DESC
LIMIT 10;
""", description="3. Folders with most files")


--- 3. Folders with most files ---



,name,path,file_count
0,documents,C:/Users/RhysL/Desktop/DE_Tools/venv/Lib/site-...,535
1,__pycache__,C:/Users/RhysL/Desktop/DE_Tools/venv/Lib/site-...,258
2,lexers,C:/Users/RhysL/Desktop/DE_Tools/venv/Lib/site-...,258
3,JMulTi_results,C:/Users/RhysL/Desktop/DE_Tools/venv/Lib/site-...,175
4,America,C:/Users/RhysL/Desktop/DE_Tools/venv/Lib/site-...,143
5,America,C:/Users/RhysL/Desktop/DE_Tools/venv/Lib/site-...,142
6,matplotlib,C:/Users/RhysL/Desktop/DE_Tools/venv/Lib/site-...,141
7,2and3,C:/Users/RhysL/Desktop/DE_Tools/venv/Lib/site-...,133
8,pyasn1_modules,C:/Users/RhysL/Desktop/DE_Tools/venv/Lib/site-...,132
9,2,C:/Users/RhysL/Desktop/DE_Tools/venv/Lib/site-...,112


In [19]:
run_query("""
SELECT extension, COUNT(*) AS count
FROM files
GROUP BY extension
ORDER BY count DESC;
""", description="4. Count files by extension")


--- 4. Count files by extension ---



,extension,count
0,.py,10150
1,.pyc,5266
2,.pyi,2272
3,,1781
4,.json,554
...,...,...
119,.bash,1
120,.ani,1
121,.PSF,1
122,.MIT,1


In [20]:
run_query("""
SELECT folders.name AS folder_name, files.name, files.path, files.size
FROM files
JOIN folders ON files.folder_id = folders.id
ORDER BY files.size DESC
LIMIT 10;
""", description="5. Largest files with folder name")


--- 5. Largest files with folder name ---



,folder_name,name,path,size
0,numpy.libs,libscipy_openblas64_-43e11ff0749b8cbe0a615c9cf...,C:/Users/RhysL/Desktop/DE_Tools/venv/Lib/site-...,20301824
1,scipy.libs,libscipy_openblas-f07f5a5d207a3a47104dca54d6d0...,C:/Users/RhysL/Desktop/DE_Tools/venv/Lib/site-...,20155392
2,database,longlist.db,C:/Users/RhysL/Desktop/DE_Tools/Explorations/S...,6709248
3,Outputs,long_list_viewing.db,C:/Users/RhysL/Desktop/DE_Tools/Explorations/S...,6709248
4,Outputs,long_list_changed.db,C:/Users/RhysL/Desktop/DE_Tools/Explorations/S...,6709248
5,_highspy,_core.cp312-win_amd64.pyd,C:/Users/RhysL/Desktop/DE_Tools/venv/Lib/site-...,5775872
6,pydevd_attach_to_process,inject_dll_x86.pdb,C:/Users/RhysL/Desktop/DE_Tools/venv/Lib/site-...,5771264
7,pydevd_attach_to_process,inject_dll_amd64.pdb,C:/Users/RhysL/Desktop/DE_Tools/venv/Lib/site-...,5656576
8,pythonwin,mfc140u.dll,C:/Users/RhysL/Desktop/DE_Tools/venv/Lib/site-...,5653576
9,documents,compute.alpha.json,C:/Users/RhysL/Desktop/DE_Tools/venv/Lib/site-...,4825049


In [21]:

run_query("""
SELECT extension, AVG(size) AS avg_size, COUNT(*) AS count
FROM files
GROUP BY extension
ORDER BY avg_size DESC;
""", description="6. Average file size by extension")


--- 6. Average file size by extension ---



,extension,avg_size,count
0,.dll,3.335653e+06,15
1,.chm,2.639762e+06,1
2,.pdb,2.491733e+06,6
3,.accdb,1.302528e+06,2
4,.sqlite,1.067008e+06,2
...,...,...,...
119,.lua,1.390000e+02,1
120,.log,5.400000e+01,1
121,.z,4.100000e+01,3
122,.inc,1.700000e+01,1


In [22]:

run_query("""
SELECT name, path, modified_time
FROM files
ORDER BY modified_time DESC
LIMIT 10;
""", description="7. Files modified recently")


--- 7. Files modified recently ---



,name,path,modified_time
0,folder_structure.db-journal,C:/Users/RhysL/Desktop/DE_Tools/Explorations/O...,2025-07-12T21:49:07.688735
1,folder_structure.db,C:/Users/RhysL/Desktop/DE_Tools/Explorations/O...,2025-07-12T21:48:58.846389
2,grep-commands.txt,C:/Users/RhysL/Desktop/DE_Tools/Explorations/O...,2025-07-12T21:42:03.155010
3,names.txt,C:/Users/RhysL/Desktop/DE_Tools/Explorations/O...,2025-07-12T21:42:00.335504
4,decorators.py,C:/Users/RhysL/Desktop/DE_Tools/Explorations/O...,2025-07-12T21:02:47.134849
5,decorators.ipynb,C:/Users/RhysL/Desktop/DE_Tools/Explorations/O...,2025-07-12T21:02:44.520004
6,snippets.txt,C:/Users/RhysL/Desktop/DE_Tools/Explorations/O...,2025-07-12T21:02:42.614765
7,display_info.log,C:/Users/RhysL/Desktop/DE_Tools/Explorations/O...,2025-07-12T21:01:14.980376
8,main.ipynb,C:/Users/RhysL/Desktop/DE_Tools/Explorations/O...,2025-07-12T20:51:13.040846
9,test_folders.csv,C:/Users/RhysL/Desktop/DE_Tools/test_folders.csv,2025-07-12T19:46:50.374962


In [36]:
# --- Queries with variables for flexible exploration ---

# Variable: Folder path prefix to filter folders (and files within)
folder_path_prefix = 'C:\Users\RhysL\Desktop\DE_Tools\Explorations\Other\Folder-Tools'  # Adjust as needed


SyntaxError: (unicode error) 'unicodeescape' codec can't decode bytes in position 2-3: truncated \UXXXXXXXX escape (1582106470.py, line 4)

In [38]:

query_folder_files = """
SELECT f.name AS folder_name, fi.name AS file_name, fi.path AS file_path
FROM folders f
JOIN files fi ON f.id = fi.folder_id
WHERE f.path LIKE ?
ORDER BY f.path, fi.name
LIMIT 10;
"""
run_query(query_folder_files, params=(folder_path_prefix + '%',), description=f"8. Files in folders starting with '{folder_path_prefix}'")


--- 8. Files in folders starting with 'C:/Users/RhysL/Desktop/DE_Tools/Explorations' ---



,folder_name,file_name,file_path
0,Cleaning,Dataframe_Cleaing.ipynb,C:/Users/RhysL/Desktop/DE_Tools/Explorations/C...
1,Cleaning,Handling_Missing_Data.ipynb,C:/Users/RhysL/Desktop/DE_Tools/Explorations/C...
2,Cleaning,Handling_Missing_Data_Basic.ipynb,C:/Users/RhysL/Desktop/DE_Tools/Explorations/C...
3,ETL,README.md,C:/Users/RhysL/Desktop/DE_Tools/Explorations/E...
4,ETL,main.ipynb,C:/Users/RhysL/Desktop/DE_Tools/Explorations/E...
5,raw,database.db,C:/Users/RhysL/Desktop/DE_Tools/Explorations/E...
6,raw,sales.csv,C:/Users/RhysL/Desktop/DE_Tools/Explorations/E...
7,raw,sales.xlsx,C:/Users/RhysL/Desktop/DE_Tools/Explorations/E...
8,schema,schema.sql,C:/Users/RhysL/Desktop/DE_Tools/Explorations/E...
9,src,README.md,C:/Users/RhysL/Desktop/DE_Tools/Explorations/E...


In [40]:
# Variable: File extension to filter
file_extension = '.md'

query_files_by_extension = """
SELECT name, path, size, modified_time
FROM files
WHERE extension = ?
ORDER BY modified_time DESC
LIMIT 10;
"""
run_query(query_files_by_extension, params=(file_extension,), description=f"9. Most recent files with extension '{file_extension}'")


--- 9. Most recent files with extension '.md' ---



,name,path,size,modified_time
0,text.md,C:/Users/RhysL/Desktop/DE_Tools/Explorations/O...,0,2025-07-12T19:46:50.368541
1,text.md,C:/Users/RhysL/Desktop/DE_Tools/Explorations/O...,0,2025-07-12T19:46:50.362760
2,ideas.md,C:/Users/RhysL/Desktop/DE_Tools/Explorations/O...,0,2025-05-20T20:36:29.986606
3,notes.md,C:/Users/RhysL/Desktop/DE_Tools/Explorations/S...,0,2025-04-28T20:41:19.450723
4,README.md,C:/Users/RhysL/Desktop/DE_Tools/Explorations/T...,74,2025-03-30T12:44:21.335425
5,Tracker.md,C:/Users/RhysL/Desktop/DE_Tools/Explorations/S...,869,2025-03-29T19:06:59.453146
6,README.md,C:/Users/RhysL/Desktop/DE_Tools/Explorations/P...,102,2025-03-29T18:25:21.748662
7,README.md,C:/Users/RhysL/Desktop/DE_Tools/README.md,965,2025-03-24T07:27:09.407579
8,LICENSE.md,C:/Users/RhysL/Desktop/DE_Tools/venv/Lib/site-...,1096,2025-03-23T16:22:24.573289
9,LICENSE.md,C:/Users/RhysL/Desktop/DE_Tools/venv/Lib/site-...,1523,2025-03-22T21:15:36.929019


In [ ]:

# Variable: File name to search exact matches
file_name_search = 'README.md'

query_files_by_name = """
SELECT name, path, size, modified_time
FROM files
WHERE name = ?
ORDER BY modified_time DESC
LIMIT 10;
"""
run_query(query_files_by_name, params=(file_name_search,), description=f"10. Files named '{file_name_search}'")


In [41]:

# Variable: Minimum file size to filter large files (bytes)
min_size = 1_000_000  # 1 MB

query_large_files = """
SELECT name, path, size
FROM files
WHERE size > ?
ORDER BY size DESC
LIMIT 10;
"""
run_query(query_large_files, params=(min_size,), description=f"11. Files larger than {min_size} bytes")


--- 11. Files larger than 1000000 bytes ---



,name,path,size
0,libscipy_openblas64_-43e11ff0749b8cbe0a615c9cf...,C:/Users/RhysL/Desktop/DE_Tools/venv/Lib/site-...,20301824
1,libscipy_openblas-f07f5a5d207a3a47104dca54d6d0...,C:/Users/RhysL/Desktop/DE_Tools/venv/Lib/site-...,20155392
2,longlist.db,C:/Users/RhysL/Desktop/DE_Tools/Explorations/S...,6709248
3,long_list_viewing.db,C:/Users/RhysL/Desktop/DE_Tools/Explorations/S...,6709248
4,long_list_changed.db,C:/Users/RhysL/Desktop/DE_Tools/Explorations/S...,6709248
5,_core.cp312-win_amd64.pyd,C:/Users/RhysL/Desktop/DE_Tools/venv/Lib/site-...,5775872
6,inject_dll_x86.pdb,C:/Users/RhysL/Desktop/DE_Tools/venv/Lib/site-...,5771264
7,inject_dll_amd64.pdb,C:/Users/RhysL/Desktop/DE_Tools/venv/Lib/site-...,5656576
8,mfc140u.dll,C:/Users/RhysL/Desktop/DE_Tools/venv/Lib/site-...,5653576
9,compute.alpha.json,C:/Users/RhysL/Desktop/DE_Tools/venv/Lib/site-...,4825049


In [42]:

# Variable: Count files grouped by year of modification
query_files_by_year = """
SELECT SUBSTR(modified_time, 1, 4) AS year, COUNT(*) AS count
FROM files
GROUP BY year
ORDER BY year DESC;
"""
run_query(query_files_by_year, description="12. Count of files by year")


--- 12. Count of files by year ---



,year,count
0,2025,22931
1,2024,40
2,2022,2
3,2021,23


In [43]:

conn.close()


## Powershell

In [ ]:
Get-ChildItem -Recurse | Where-Object {!$_.PSIsContainer} |
  Select-Object FullName, Length, LastWriteTime

redo without python


Get-ChildItem -Path "C:\Your\Target\Directory" -Recurse -Force |
  Where-Object { !$_.Attributes.ToString().Contains("Hidden") -and !$_.Attributes.ToString().Contains("System") } |
  Select-Object FullName, Name, Extension, Length, LastWriteTime, CreationTime |
  Export-Csv -Path "visible_files.csv" -NoTypeInformation


C:\Users\RhysL\Desktop\DE_Tools



# Set target directory
$targetPath = "C:\Users\RhysL\Desktop\DE_Tools"

# Set output CSV path
$outputCsv = "C:\Users\RhysL\Desktop\DE_Tools\Explorations\Other\Folder-Tools\Folder-DB\visible_files.csv"

# Run filtered export
Get-ChildItem -Path $targetPath -Recurse -Force |
  Where-Object {
    -not $_.Attributes.ToString().Contains("Hidden") -and
    -not $_.Attributes.ToString().Contains("System") -and
    -not $_.PSIsContainer
  } |
  Select-Object FullName, Name, Extension, Length, LastWriteTime, CreationTime |
  Export-Csv -Path $outputCsv -NoTypeInformation
